# Fine‑tune FLAN-T5 on Instruction Data

**Author:** Ibrahim  
**Environment:** Google Colab / T4 GPU (optional)

## Overview
This notebook fine‑tunes a small, instruction‑tuned model (FLAN‑T5‑Small) on a subset of the Databricks Dolly 15k dataset. It demonstrates the complete fine‑tuning pipeline using Hugging Face `Trainer` – no complex libraries, guaranteed to run.

## Metrics
- Training loss
- Sample inference after fine‑tuning

**© 2026 Ibrahim – End‑to‑end LLM fine‑tuning.**

### Install Dependencies


In [13]:
!pip install -q transformers datasets accelerate sentencepiece

### Imports & Model Loading

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on cuda


### Load & Format Dataset

In [15]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.select(range(500))  # first 500 examples

def format_instruction(example):
    return {
        "input_text": f"Instruction: {example['instruction']}\nResponse:",
        "target_text": example['response']
    }

dataset = dataset.map(format_instruction)

def tokenize(batch):
    model_inputs = tokenizer(batch["input_text"], truncation=True, padding="max_length", max_length=128)
    labels = tokenizer(batch["target_text"], truncation=True, padding="max_length", max_length=128)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["instruction", "response", "input_text", "target_text"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Trainer Setup

In [16]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="./flan-t5-dolly",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

### Train

In [17]:
trainer.train()
print("Training complete.")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 76,961,152 of 76,961,152 (100.00% trained)


Step,Training Loss
50,0.000000


Training complete.


### Save Model

In [18]:
model.save_pretrained("flan-t5-finetuned")
tokenizer.save_pretrained("flan-t5-finetuned")
print("Model saved to ./flan-t5-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./flan-t5-finetuned


### Inference Test

In [19]:
def generate(prompt):
    input_text = f"Instruction: {prompt}\nResponse:"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    "What is the capital of France?",
    "Explain machine learning in simple terms.",
]

for p in test_prompts:
    print(f" {p}")
    print(f" {generate(p)}\n")

 What is the capital of France?
 arrondissements of France

 Explain machine learning in simple terms.
 Using a computer, you can learn a machine learning style by using a computer.

